# Visualizacoes das Variaveis — Camada Silver (AV1)

Este notebook complementa a demonstracao tecnica apresentando **visualizacoes exploratorias** das variaveis da camada silver (`data/silver/sihsus_sp.parquet`).

**Escopo:**
- Distribuicao de nulos por coluna
- Distribuicoes univariadas (numericas e categoricas)
- Serie temporal bruta de volume de internacoes
- Perfil demografico (sexo, faixa etaria)
- Flag COVID ao longo do tempo

> **Importante:** nenhuma agregacao e persistida em disco. As visualizacoes sao calculadas em memoria sobre a silver granular (1 linha = 1 AIH). Agregacoes e analises consolidadas sao escopo da AV2.

## 0. Configuracao

In [ ]:
import os
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

ROOT = Path(os.getcwd())
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

SILVER_FILE = ROOT / 'data' / 'silver' / 'sihsus_sp.parquet'
print(f'Silver: {SILVER_FILE} | existe: {SILVER_FILE.exists()}')

In [ ]:
df = pd.read_parquet(SILVER_FILE)
print(f'Registros: {len(df):,}')
print(f'Colunas:   {len(df.columns)}')
print(f'Periodo:   {df["DT_INTER"].min().date()} a {df["DT_INTER"].max().date()}')
df.head(3)

## 1. Volume por ano

A silver nao possui nulos (garantido por `drop_nulls` no pipeline). Esta visao confirma a cobertura temporal completa 2020-2023.

In [ ]:
assert df.isnull().sum().sum() == 0, 'Silver nao deveria conter nulos'

volume_ano = df['ano'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
volume_ano.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Internacoes por ano — SIH/SUS SP')
ax.set_xlabel('Ano')
ax.set_ylabel('Contagem')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(volume_ano.values):
    ax.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 2. Distribuicoes univariadas — Numericas

In [ ]:
num_cols = ['DIAS_PERM', 'UTI_MES_TO', 'IDADE', 'VAL_TOT']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, num_cols):
    serie = pd.to_numeric(df[col], errors='coerce').dropna()
    q99 = serie.quantile(0.99)  # corta cauda longa para visualizacao
    ax.hist(serie[serie <= q99], bins=50, color='steelblue', edgecolor='white')
    ax.set_title(f'{col} (ate p99)')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequencia')
plt.tight_layout()
plt.show()

## 3. Distribuicoes univariadas — Categoricas

In [ ]:
cat_cols = ['SEXO', 'CAR_INT', 'COMPLEX']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, cat_cols):
    counts = df[col].fillna('NA').value_counts().head(10)
    counts.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(col)
    ax.set_xlabel('')
    ax.set_ylabel('Contagem')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 4. Serie temporal — Volume mensal de internacoes

Contagem mensal de linhas. Este nao e um Parquet agregado persistido — e apenas um `value_counts` em memoria para visualizar a variavel `ano_mes`.

In [ ]:
volume_mensal = df['ano_mes'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
volume_mensal.plot(ax=ax, color='steelblue', linewidth=2)
ax.set_title('Volume mensal de internacoes SIH/SUS — SP (2020-2023)')
ax.set_xlabel('Mes')
ax.set_ylabel('Internacoes')
plt.tight_layout()
plt.show()

## 5. Perfil demografico — Faixa etaria e sexo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df['faixa_etaria'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Internacoes por faixa etaria')
axes[0].set_xlabel('Faixa etaria')
axes[0].set_ylabel('Contagem')
axes[0].tick_params(axis='x', rotation=0)

df['SEXO'].fillna('NA').value_counts().plot(
    kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Internacoes por sexo')
axes[1].set_xlabel('Sexo (1=M, 3=F)')
axes[1].set_ylabel('Contagem')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 6. Flag COVID — distribuicao ao longo do tempo

In [ ]:
covid_mensal = (
    df.groupby('ano_mes')['is_covid']
      .sum()
      .sort_index()
)

fig, ax = plt.subplots(figsize=(12, 4))
covid_mensal.plot(ax=ax, color='crimson', linewidth=2)
ax.set_title('Internacoes COVID (CID B342) por mes — SP')
ax.set_xlabel('Mes')
ax.set_ylabel('Internacoes COVID')
plt.tight_layout()
plt.show()

---
**Observacao final:** todas as visualizacoes acima sao calculadas em memoria sobre a silver granular. Nenhuma tabela agregada e gravada em `data/`. A geracao de tabelas agregadas e das analises consolidadas (mortalidade por onda, UTI, impacto regional) e escopo da AV2.